# 52. 수정 검색 비교의 원문 판정·시간·비용 집계
유료 실행 노트북의 동결 코드는 그대로 두고 저장된 결과만 검사한다. API 호출·모델 로드·HTML 생성은 하지 않는다. 전체280답변과 원문판정이 없으면 집계를 거절한다. HTML은 이 출력과 원문을 확인한 뒤 직접 작성한다. 원98판정과 사후 해석 정정은 함께 공개하며 독립 인간 평가로 부르지 않는다.

카드 검색 지표의37문항은 정답 근거가 있는 카드 문서가 존재하는 질문이다. 추천 질문은 실제 조건을 충족하는 카드, 적용불가 확인 질문은 올바른 거절 근거를 가진 해당 카드로 판정한다. 후자를 혜택 적용 가능 카드로 오해하지 않는다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,math,statistics,collections
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('analysis52',ROOT/'notebooks/scripts/51_experiment_runtime.py')
X=importlib.util.module_from_spec(spec);sys.modules[spec.name]=X;spec.loader.exec_module(X)
X.load('analysis_recovery52','51_budget_recovery.py').install(X)
X.load('analysis_corrected52','52_query_predicate_correction.py').install(X)
P=X.P;PATH=P.F.STAGES[52];E=X.load('analysis_question_eval52','53_question_evaluation.py')
manifest=P.F.read_json(PATH/'execution_manifest.json');X.guard_manifest(manifest)
completed=P.F.read_json(PATH/'execution_results.json');assert completed['manifest_sha256']==P.sha(manifest)
rows=completed['answers'];retrievals=completed['retrievals'];annotations=P.F.read_json(PATH/'question_source_judgments.json')
judged={j['identity']:j for j in annotations['records']};retrieved={r['identity']:r for r in retrievals}
assert len(rows)==len(judged)==len(annotations['records'])==len(retrievals)==280
assert set(judged)==set(retrieved)=={r['identity'] for r in rows}
queries=P.F.read_json(PATH/'questions.json')['queries'];qmap={q['query_id']:q for q in queries}
rubrics=P.F.read_json(PATH/'question_rubrics.json');rubric_by_query={r['query_id']:r for r in rubrics['records']}
supplement=P.F.read_json(PATH/'evaluation_interpretation_supplement.json')
assert annotations['evaluation_interpretation_sha256']==P.sha(supplement)
import fitz
page_counts={}
for row in rows:
 j=judged[row['identity']];q=qmap[row['query_id']]
 assert j['answer_sha256']==P.sha(row) and j['question_sha256']==P.sha(q['question']) and j['reason']
 assert rubric_by_query[q['query_id']]['question']==q['question']
 E.validate_judgment(j,rubric_by_query[q['query_id']],P.sha)
 assert j['source_support'] and (row['status']=='ok' or j['question_verdict']!='PASS')
 for s in j['source_support']:
  if s.get('kind')=='corpus_inventory':
   assert q['subgroup']=='negative_diagnostic' and not rubric_by_query[q['query_id']]['gold_cards']
   assert s['documents_sha256']==P.sha(X.Q.documents()) and s['observation']
   continue
  assert s['path']=='data/raw/'+s['card_key']+'.pdf' and P.sha((ROOT/s['path']).read_bytes())==s['sha256']
  if s['path'] not in page_counts:page_counts[s['path']]=len(fitz.open(ROOT/s['path']))
  assert type(s['page']) is int and 1<=s['page']<=page_counts[s['path']] and s['observation']
catalog={}
for name in ['anchor_source_judgments.json','verified_additional_cards.json']:
 for r in P.F.read_json(PATH/'source_review'/name)['records']:catalog[r['query_id'],r['card_key']]=r
for c in catalog.values():
 assert type(c['valid_for_original_question']) is bool and c['reason'] and c['source_support']
 for support in c['source_support']:
  assert support['path']=='data/raw/'+c['card_key']+'.pdf' and P.sha((ROOT/support['path']).read_bytes())==support['sha256']
  if support['path'] not in page_counts:page_counts[support['path']]=len(fitz.open(ROOT/support['path']))
  assert type(support['page']) is int and 1<=support['page']<=page_counts[support['path']] and support['observation']
valid={q['query_id']:{k for (qid,k),c in catalog.items() if qid==q['query_id'] and c['valid_for_original_question']} for q in queries}
source_questions=[q['query_id'] for q in queries if valid[q['query_id']]];assert len(source_questions)==37
print('280개 전체 답변과 원문·질문·판정 결속 확인. 카드 검색 지표의 고정 질문 분모37.')

280개 전체 답변과 원문·질문·판정 결속 확인. 카드 검색 지표의 고정 질문 분모37.


In [2]:
def describe(values):
 values=sorted(values);n=len(values)
 return dict(n=n,mean=statistics.mean(values),median=statistics.median(values),p95=values[math.ceil(.95*n)-1],maximum=values[-1]) if n else dict(n=0,mean=None,median=None,p95=None,maximum=None)

def body(result):return {k:result.get(k) for k in ['cards','evidence','clarify']}
configs=[('common','S_ONLY')]+[(pf,m) for pf in X.PROFILES for m in ['U','SEQ','PAR']]
lookup={(r['query_id'],r['profile'],r['mode']):r for r in rows}
Q_ALL={d['document_id'] for d in X.Q.documents()};assert len(Q_ALL)==106
quality=[];pairs=[];timing=[]
for pf,mode in configs:
 rr=[r for r in rows if r['profile']==pf and r['mode']==mode];assert len(rr)==40
 counts=collections.Counter(judged[r['identity']]['question_verdict'] for r in rr)
 hits=[];reciprocal=[];subgroups={};physical={};coverage=collections.Counter()
 for r in rr:
  q=qmap[r['query_id']];j=judged[r['identity']];retr=retrieved[r['identity']]
  group=subgroups.setdefault(q['subgroup'],dict(n=0,PASS=0));group['n']+=1;group['PASS']+=j['question_verdict']=='PASS'
  for field,values in j['annotation_coverage'].items():coverage[field]+=len(values)
  for event in r['events']:
   record=P.F.read_json(PATH/'api_cache'/(event['key']+'.json'));physical[event['key']]=record
  if r['query_id'] in source_questions:
   good=[i+1 for i,c in enumerate(retr['result']['cards'][:3]) if c['card_key'] in valid[r['query_id']]]
   hits.append(bool(good));reciprocal.append(1/min(good) if good else 0)
 item=dict(profile=pf,mode=mode,logical_denominator=40,verdict_counts=dict(counts),subgroups=subgroups,
  important_errors=sum(judged[r['identity']]['important_error'] for r in rr),scope_errors=sum(judged[r['identity']]['scope_error'] for r in rr),
  hit_denominator=37,Hit_at_3=sum(hits)/37,MRR_at_3=sum(reciprocal)/37,hit_count=sum(hits),
  annotation_coverage_counts=dict(coverage),annotation_coverage_rate=coverage['covered']/(coverage['covered']+coverage['missing']) if coverage['covered']+coverage['missing'] else None)
 quality.append(item)
 physical_rows=list(physical.values())
 physical_cost=P.PRICE.aggregate(P.PRICE.price_records([('52',str((PATH/'api_cache'/(k+'.json')).relative_to(ROOT)),v) for k,v in physical.items()]))
 timing.append(dict(profile=pf,mode=mode,answer_event_usage_based_cost=physical_cost,cost_scope='answer events only; parser and embedding excluded',answer_ready_seconds=describe([r['answer_ready_wall_seconds'] for r in rr]),
  physical_API_seconds=describe([r['wall_seconds'] for r in physical_rows]),physical_API_requests=len(physical_rows),
  technical_answer_failures=sum(r['status']!='ok' for r in rr),validation_retry_answers=sum(len(r['events'])>1 for r in rr),
  first_write_search_seconds=describe([retrieved[r['identity']]['result']['experiment_timing']['search_wall_seconds'] for r in rr if 'experiment_timing' in retrieved[r['identity']]['result']]),
  observed_retrieval_seconds=describe([retrieved[r['identity']]['observed_retrieval_wall_seconds'] for r in rr if 'observed_retrieval_wall_seconds' in retrieved[r['identity']]]),
  timing_caution='U first search is historical; same-input retrieval shares first timing; observed lookup is cache replay when reuse is set; no measured online end-to-end/TTFT/UI'))
for pf in X.PROFILES:
 for mode in ['S_ONLY','SEQ','PAR']:
  comparisons=[]
  for q in queries:
   uid=lookup[q['query_id'],pf,'U']['identity'];r=lookup[q['query_id'],'common' if mode=='S_ONLY' else pf,mode];cid=r['identity']
   u=judged[uid]['question_verdict']=='PASS';c=judged[cid]['question_verdict']=='PASS'
   seq=retrieved[lookup[q['query_id'],pf,'SEQ']['identity']];scope=seq['route'].get('scope');original=seq['original_scope']['card_keys']
   narrowed=len(scope if scope is not None else Q_ALL)<len(original if original is not None else Q_ALL)
   comparisons.append(dict(query_id=q['query_id'],U_verdict=judged[uid]['question_verdict'],candidate_verdict=judged[cid]['question_verdict'],outcome='win' if c and not u else 'loss' if u and not c else 'tie',
    actual_scope_reduction=narrowed if mode=='SEQ' else None,SQL_matched=(seq['route']['reason']=='matched') if mode=='SEQ' else None,same_U_SEQ_body=(body(retrieved[uid]['result'])==body(seq['result'])) if mode=='SEQ' else None))
  pairs.append(dict(profile=pf,mode=mode,question_denominator=40,question_wins_losses_ties=dict(collections.Counter(r['outcome'] for r in comparisons)),records=comparisons))
print(json.dumps(quality,ensure_ascii=False,indent=2))

[
  {
    "profile": "common",
    "mode": "S_ONLY",
    "logical_denominator": 40,
    "verdict_counts": {
      "PARTIAL": 5,
      "PASS": 16,
      "FAIL": 19
    },
    "subgroups": {
      "scope": {
        "n": 3,
        "PASS": 2
      },
      "detail": {
        "n": 1,
        "PASS": 1
      },
      "numeric": {
        "n": 10,
        "PASS": 4
      },
      "semantic": {
        "n": 10,
        "PASS": 1
      },
      "proper_noun": {
        "n": 8,
        "PASS": 3
      },
      "negative_diagnostic": {
        "n": 8,
        "PASS": 5
      }
    },
    "important_errors": 4,
    "scope_errors": 0,
    "hit_denominator": 37,
    "Hit_at_3": 0.5945945945945946,
    "MRR_at_3": 0.5945945945945946,
    "hit_count": 22,
    "annotation_coverage_counts": {
      "covered": 14,
      "missing": 5,
      "not_applicable": 0,
      "uncertain": 126
    },
    "annotation_coverage_rate": 0.7368421052631579
  },
  {
    "profile": "card_page_section_benefit",
    "mode

In [3]:
def stage_cost(stages):
 records=[(str(stage),str(p.relative_to(ROOT)),P.F.read_json(p)) for stage in stages for p in sorted((P.F.STAGES[stage]/'api_cache').glob('*.json'))]
 return P.PRICE.aggregate(P.PRICE.price_records(records))
# 모든 원문 판정을 마친 뒤 기계 판독용 집계만 저장한다. HTML은 직접 작성한다.
summary=dict(analysis_code_sha256=X.notebook_code_hash(ROOT/'notebooks/52_corrected_four_retrieval_modes_evaluation.ipynb'),evaluation_helper_sha256=P.sha((ROOT/'notebooks/scripts/53_question_evaluation.py').read_bytes()),split='development',primary_metric='question_verdict',secondary_metric='annotation_coverage',logical_denominator=280,
 common_S_ONLY_conditions=40,configurations=quality,paired_comparisons=pairs,timing=timing,
 execution_results_sha256=P.sha(completed),source_judgments_sha256=P.sha(annotations),question_rubrics_sha256=P.sha(rubrics),
 evaluation_interpretation_sha256=P.sha(supplement),posthoc_development_clarification=True,independent_human_review=False,production_adopted=False,
 conditional_ranking_review=P.F.read_json(PATH/'conditional_trigger_review.json'),stage52_current_cost=stage_cost([52]),preserved_original52_cost=stage_cost([520]),stage52_combined_cost=stage_cost([52,520]),cumulative_authorized_experiment_cost=X.ledger(),cost_scope_note='stage52 combines corrected52 plus preserved original52 once; config cost contains answer events only; cumulative ledger includes previous stages and authorized retry')
X.frozen(PATH/'evaluation_summary.json',summary)
print('개발 평가 집계 저장. 운영 선정은 별도 확인시험 전이다.')

개발 평가 집계 저장. 운영 선정은 별도 확인시험 전이다.
